In [ ]:
# ============================================================
# CELL 1 - TITLE AND INSTRUCTIONS
# ============================================================
"""
╔══════════════════════════════════════════════════════════╗
║     BUSINESS AI DATASET GENERATOR                       ║
║     Sprint 1: Synthetic Data Generation                 ║
╠══════════════════════════════════════════════════════════╣
║  HOW TO USE THIS NOTEBOOK:                              ║
║                                                         ║
║  1. Add secrets in Colab:                               ║
║     Sidebar → 🔑 Secrets → Add each variable            ║
║     DB_USER, DB_PASSWORD, DB_HOST,                      ║
║     DB_NAME, ACCOUNT_ID                                 ║
║                                                         ║
║  2. Run Cell 1 (Install)                                ║
║  3. Run Cell 2 (Setup)                                  ║
║  4. Run Cell 3 (Database)                               ║
║  5. Choose your model in Cell 4                         ║
║  6. Run Cell 5 (Generate)                               ║
║                                                         ║
║  Session will auto-checkpoint every 10 samples.         ║
║  If session dies → restart and resume automatically.    ║
╚══════════════════════════════════════════════════════════╝
"""

# ============================================================
# CELL 2 - INSTALL DEPENDENCIES
# Run this first every session
# Takes about 3-5 minutes
# ============================================================

!pip install -q \
    transformers==4.44.0 \
    bitsandbytes==0.43.3 \
    accelerate==0.33.0 \
    psycopg2-binary==2.9.9 \
    sqlalchemy==2.0.31 \
    pgvector==0.3.2 \
    torch==2.3.1 \
    sentencepiece \
    protobuf

print("✅ All dependencies installed")

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU: {gpu} | VRAM: {vram:.1f}GB")
else:
    print("⚠️  No GPU detected. Generation will be very slow.")
    print("   Go to Runtime → Change runtime type → T4 GPU")


# ============================================================
# CELL 3 - CLONE REPO AND SET CREDENTIALS
# ============================================================

import os
from google.colab import userdata

# Clone the repository
!git clone https://github.com/YOUR_USERNAME/business-ai-datagen.git
%cd business-ai-datagen

# Load credentials from Colab Secrets
# Add these in Colab sidebar → 🔑 icon → Secrets
os.environ["DB_HOST"]     = userdata.get("DB_HOST")
os.environ["DB_PORT"]     = userdata.get("DB_PORT")
os.environ["DB_NAME"]     = userdata.get("DB_NAME")
os.environ["DB_USER"]     = userdata.get("DB_USER")
os.environ["DB_PASSWORD"] = userdata.get("DB_PASSWORD")
os.environ["DB_SSLMODE"]  = "require"

# Set account ID for this Colab account
# Change this for each parallel account:
# ACCOUNT-1 → ACCOUNT-2 → ACCOUNT-3
os.environ["ACCOUNT_ID"] = userdata.get(
    "ACCOUNT_ID", "ACCOUNT-1"
)

print("✅ Environment variables set")
print(f"✅ Account ID: {os.environ['ACCOUNT_ID']}")
print(f"✅ DB Host   : {os.environ['DB_HOST']}")


# ============================================================
# CELL 4 - INITIALIZE DATABASE
# Run once per session
# Verifies connection and tables
# ============================================================

import sys
sys.path.append("/content/business-ai-datagen")

from src.database.setup import initialize_database
from src.prompts.categories import print_category_plan

# Initialize and verify DB connection
db = initialize_database()

# Print the full generation plan
print_category_plan()


# ============================================================
# CELL 5 - CHOOSE YOUR MODEL
# ─────────────────────────────────────────────────────────
# IMPORTANT: Each Colab account should run ONE model
#
# Account 1 → model_name = "qwen"
# Account 2 → model_name = "deepseek"
# Account 3 → model_name = "phi"
# ============================================================

# ↓↓↓ CHANGE THIS BASED ON YOUR ACCOUNT ↓↓↓
model_name = "qwen"
# ↑↑↑ OPTIONS: "qwen" | "deepseek" | "phi" ↑↑↑

# How many samples to generate this session
# 200 is safe for a 4-hour Colab Free session
max_samples = 200

print(f"✅ Model selected   : {model_name}")
print(f"✅ Session target   : {max_samples} samples")
print(f"✅ Account ID       : {os.environ['ACCOUNT_ID']}")
print()
print("▶️  Run Cell 6 to start generation")


# ============================================================
# CELL 6 - START OR RESUME GENERATION
# ─────────────────────────────────────────────────────────
# This cell runs the full pipeline.
# It will:
#   1. Check if there is a paused batch to resume
#   2. Load the selected model (takes 15-20 mins)
#   3. Generate samples with self-critique
#   4. Save every sample to GCP PostgreSQL
#   5. Checkpoint every 10 samples
#   6. Show dashboard every 50 samples
#   7. Auto-pause cleanly if session nears 4hr limit
# ============================================================

from src.pipeline.orchestrator import GenerationOrchestrator

orchestrator = GenerationOrchestrator(
    model_name=model_name,
    account_id=os.environ.get("ACCOUNT_ID", "ACCOUNT-1")
)

orchestrator.run(max_samples=max_samples)


# ============================================================
# CELL 7 - CHECK PROGRESS (Run anytime)
# Shows current dataset stats without
# interrupting generation
# ============================================================

from src.database.checkpoint import CheckpointManager
from src.utils.dashboard import Dashboard
from datetime import datetime

ckpt      = CheckpointManager(
    os.environ.get("ACCOUNT_ID", "ACCOUNT-1")
)
dashboard = Dashboard()
dashboard.show(ckpt, orchestrator.writer, datetime.now())


# ============================================================
# CELL 8 - EXPORT DATASET TO JSONL
# Run this after generation to export
# clean training-ready dataset
# ============================================================

import json
from src.database.setup import get_db

def export_to_jsonl(
    output_path: str = "/content/business_ai_dataset.jsonl",
    quality_filter: str = "PASS",
    limit: int = None
):
    """
    Export filtered dataset to JSONL format.
    Ready for Sprint 2 QLoRA training.
    """
    db = get_db()

    query = """
        SELECT
            sample_id,
            instruction,
            full_output,
            category,
            subcategory,
            source_model,
            quality_score,
            has_india_context,
            geography,
            industry,
            business_stage
        FROM training_samples
        WHERE quality_flag = %s
        ORDER BY quality_score DESC
    """

    params = [quality_filter]
    if limit:
        query += " LIMIT %s"
        params.append(limit)

    query += ";"

    results = db.execute_query(query, tuple(params), fetch=True)

    if not results:
        print("No results found.")
        return

    count = 0
    with open(output_path, "w") as f:
        for row in results:
            record = dict(row)

            # Format for training
            training_record = {
                "instruction": record["instruction"],
                "output":      record["full_output"],
                "metadata": {
                    "sample_id":        record["sample_id"],
                    "category":         record["category"],
                    "subcategory":      record["subcategory"],
                    "source_model":     record["source_model"],
                    "quality_score":    float(record["quality_score"]),
                    "has_india_context":record["has_india_context"],
                    "geography":        record["geography"],
                    "industry":         record["industry"],
                    "business_stage":   record["business_stage"],
                }
            }

            f.write(json.dumps(training_record) + "\n")
            count += 1

    print(f"✅ Exported {count:,} samples to {output_path}")
    print(f"   Quality filter : {quality_filter}")
    print(f"   File size      : "
          f"{os.path.getsize(output_path) / 1024 / 1024:.1f} MB")
    return output_path


# Run export
export_to_jsonl(
    output_path="/content/business_ai_dataset.jsonl",
    quality_filter="PASS"
)


# ============================================================
# CELL 9 - QUICK STATS QUERY (Run anytime)
# ============================================================

from src.database.setup import get_db

def print_quick_stats():
    """Print quick dataset statistics."""
    db = get_db()

    # Overall counts
    overall = db.execute_query("""
        SELECT
            COUNT(*)                                     as total,
            COUNT(*) FILTER (WHERE quality_flag='PASS')  as passed,
            COUNT(*) FILTER (WHERE quality_flag='FAIL')  as failed,
            COUNT(*) FILTER (WHERE quality_flag='REVIEW') as review,
            ROUND(AVG(quality_score)::numeric, 2)        as avg_score,
            ROUND(AVG(word_count)::numeric, 0)           as avg_words,
            COUNT(*) FILTER (WHERE has_india_context)    as india_samples
        FROM training_samples;
    """, fetch=True)

    # Per model counts
    by_model = db.execute_query("""
        SELECT
            source_model,
            COUNT(*)                                    as total,
            ROUND(AVG(quality_score)::numeric, 2)       as avg_score,
            COUNT(*) FILTER (WHERE quality_flag='PASS') as passed
        FROM training_samples
        GROUP BY source_model
        ORDER BY total DESC;
    """, fetch=True)

    # Per category counts
    by_category = db.execute_query("""
        SELECT
            category,
            COUNT(*) as total,
            ROUND(AVG(quality_score)::numeric, 2) as avg_score
        FROM training_samples
        GROUP BY category
        ORDER BY total DESC;
    """, fetch=True)

    # Print results
    print("\n" + "="*55)
    print("  DATASET QUICK STATS")
    print("="*55)

    if overall:
        row = dict(overall[0])
        print(f"\n  Total Samples : {row['total']:,}")
        print(f"  Passed        : {row['passed']:,}")
        print(f"  Failed        : {row['failed']:,}")
        print(f"  Review        : {row['review']:,}")
        print(f"  Avg Score     : {row['avg_score']}")
        print(f"  Avg Words     : {row['avg_words']}")
        print(f"  India Context : {row['india_samples']:,} samples")

    if by_model:
        print(f"\n  BY MODEL:")
        print(f"  {'Model':<20} {'Total':>8} {'Passed':>8} {'AvgScore':>10}")
        print("  " + "-"*50)
        for row in by_model:
            r = dict(row)
            print(
                f"  {r['source_model']:<20}"
                f" {r['total']:>8,}"
                f" {r['passed']:>8,}"
                f" {r['avg_score']:>10}"
            )

    if by_category:
        print(f"\n  BY CATEGORY:")
        print(f"  {'Category':<30} {'Total':>8} {'AvgScore':>10}")
        print("  " + "-"*52)
        for row in by_category:
            r = dict(row)
            print(
                f"  {r['category']:<30}"
                f" {r['total']:>8,}"
                f" {r['avg_score']:>10}"
            )

    print("="*55 + "\n")


print_quick_stats()